In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle
import re
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.split()

def build_vocab(texts, max_words=5000):
    counter = Counter()
    for t in texts:
        counter.update(clean_text(t))
    vocab = {"<pad>": 0, "<unk>": 1}
    for i, (word, _) in enumerate(counter.most_common(max_words - 2), start=2):
        vocab[word] = i
    return vocab

def encode(text, vocab, max_len=100):
    ids = [vocab.get(token, vocab["<unk>"]) for token in clean_text(text)][:max_len]
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    return ids


In [ ]:
df = pd.read_csv("data/dataset-exemplos.csv", sep=";")

# Usamos o mesmo LabelEncoder gerado no notebook do Numpy para consistência
with open("modelo_numpy_artefactos.pkl", "rb") as f:
    le = pickle.load(f)["label_encoder"]
labels_idx = le.transform(df["Label"])

vocab = build_vocab(df["Text"].values)

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=100):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        x = encode(self.texts[idx], self.vocab, self.max_len)
        return torch.tensor(x, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_loader = DataLoader(TextDataset(df["Text"].values, labels_idx, vocab), batch_size=16, shuffle=True)

In [4]:
# arquitetura GRU

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(hidden))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

best_loss = float('inf')

model.train()
for epoch in range(25): # Podem aumentar para 25 épocas
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")
    
    # Guarda o modelo apenas se for o melhor até agora
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "modelo_pytorch_gru.pth")
        print("  -> Novo melhor modelo guardado!")